In [8]:
import os
# Bloqueia a GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')

import numpy as np
import import_images
from csbdeep.utils import normalize
from stardist.models import StarDist2D
import matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries
from skimage.measure import regionprops, label

# ===== Parâmetro: área mínima em pixels =====
min_area = 50  # ajuste aqui

# Carrega o modelo pré-treinado
model = StarDist2D.from_pretrained('2D_versatile_fluo')

# Caminho da imagem
image_path = "/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/001024-1-001001001.tif"

# "Dicionário temporário" para usar sua função
image_paths = {0: image_path}

# Carrega a imagem
image = import_images.carregar_imagem_por_indice(image_paths, 0)
if image is None:
    raise RuntimeError("Falha ao carregar a imagem.")

print(f"Imagem carregada: {image_path} - Dimensão: {image.shape}")

# Predição do modelo (usa imagem normalizada para o modelo)
labels, details = model.predict_instances(normalize(image))

# ---------- Filtra por tamanho mínimo ----------
labels_filtrados = np.zeros_like(labels)
for prop in regionprops(labels):
    if prop.area >= min_area:
        labels_filtrados[labels == prop.label] = prop.label

# ---------- Cria overlay SOMENTE com contorno vermelho ----------
# Normaliza a imagem para [0,1]
img_float = image.astype(np.float32)
img_float -= img_float.min()
maxv = img_float.max()
if maxv > 0:
    img_float /= maxv

# Converte para RGB
image_rgb = np.stack([img_float, img_float, img_float], axis=-1)

# Acha as bordas
contornos = find_boundaries(labels_filtrados, mode='outer')

# Aplica cor vermelha nas bordas
overlay_contorno = image_rgb.copy()
overlay_contorno[contornos] = np.array([1.0, 0.0, 0.0], dtype=np.float32)

# Cria a pasta de saída
output_dir = os.path.join(os.getcwd(), "resultados", "stardist")
os.makedirs(output_dir, exist_ok=True)

# Nome base do arquivo
nome_original = os.path.splitext(os.path.basename(image_path))[0]

# Salva overlay
contorno_path = os.path.join(output_dir, f"{nome_original}_contorno.png")
plt.imsave(contorno_path, overlay_contorno)
print(f"Overlay com contorno vermelho salvo em: {contorno_path}")

# Salva subplot
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(image, cmap="gray")
ax[0].axis("off")
ax[0].set_title("Input image")

ax[1].imshow(overlay_contorno)
ax[1].axis("off")
ax[1].set_title(f"Contours > {min_area}px")

subplot_path = os.path.join(output_dir, f"{nome_original}_subplot_contorno.png")
plt.savefig(subplot_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Subplot salvo em: {subplot_path}")


Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
Imagem carregada: /home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/001024-1-001001001.tif - Dimensão: (1024, 1360)
Imagem carregada: /home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/001024-1-001001001.tif - Dimensão: (1024, 1360)
Overlay com contorno vermelho salvo em: /home/kayllany.oliveira/remote-repos/CellViability/resultados/stardist/001024-1-001001001_contorno_min50.png
Subplot salvo em: /home/kayllany.oliveira/remote-repos/CellViability/resultados/stardist/001024-1-001001001_subplot_contorno_min50.png
